# Time Series Designer

This notebook builds a representative lifecycle time series by copying and splicing two measured drive cycles (Autocross + Endurance).

## Import data

In [18]:
import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid

Autocross = pd.read_csv('EV26_Autocross_time_torque_rpm.csv')
Endurance = pd.read_csv('EV26_Endurance_time_torque_rpm.csv')

display(Autocross.head())
display(Endurance.head())

,time,torque,rpm
0,1.745773e+09,190.7500,315.0
1,1.745773e+09,194.7500,337.0
2,1.745773e+09,196.1875,365.0
3,1.745773e+09,196.1875,390.0
4,1.745773e+09,196.1875,415.0


,time,torque,rpm
0,1.746111e+09,27.0,52.0
1,1.746111e+09,28.0,51.0
2,1.746111e+09,27.0,52.0
3,1.746111e+09,28.0,61.0
4,1.746111e+09,27.0,60.0


## RPM → distance helpers

In [19]:
SECONDS_PER_HOUR = 3600.0

def elapsed_seconds(df: pd.DataFrame, time_col: str = 'time') -> float:
    t = df[time_col].to_numpy(dtype=float)
    if t.size < 2:
        return 0.0
    return float(t[-1] - t[0])


def rpm2distance(timeseries: pd.DataFrame, wheel_radius: float = 0.20574, gear_ratio: float = 2.14) -> np.ndarray:
    """
    Integrate RPM over time to compute cumulative distance traveled [m].

    Assumptions:
    - `timeseries['time']` is seconds (or any monotonic time base in seconds)
    - `timeseries['rpm']` is motor RPM
    - `gear_ratio` maps motor rotations to wheel rotations

    Returns: numpy array of cumulative distance in meters at each row.
    """
    seconds = timeseries['time'].to_numpy(dtype=float)
    rpm = timeseries['rpm'].to_numpy(dtype=float)

    rps = rpm / 60.0
    meters_per_rotation = (2.0 * np.pi * wheel_radius) / gear_ratio

    # cumulative_trapezoid integrates y(t) dt; here y=rps; multiply by meters/rotation
    dist_m = cumulative_trapezoid(rps, seconds, initial=0.0) * meters_per_rotation
    return dist_m


def cycle_summary(df: pd.DataFrame, name: str) -> None:
    d_km = float(rpm2distance(df)[-1]) / 1000.0
    h = elapsed_seconds(df) / SECONDS_PER_HOUR
    print(f'{name}: {d_km:.3f} km, {h:.3f} hours, {len(df)} rows')


cycle_summary(Autocross, 'Autocross')
cycle_summary(Endurance, 'Endurance')

Autocross: 14.898 km, 0.960 hours, 9588 rows
Endurance: 47.850 km, 1.274 hours, 98556 rows


## Create synthetic (representative) lifecycle time series

In [20]:
# Parameters
lifecycle_distance_km = 500.0
percent_autocross = 0.4
percent_endurance = 0.6

# sanity
assert lifecycle_distance_km > 0
assert 0 <= percent_autocross <= 1
assert 0 <= percent_endurance <= 1

In [21]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple


@dataclass(frozen=True)
class CycleInfo:
    df: pd.DataFrame
    distance_km: float
    duration_s: float


def _prepare_cycle(df: pd.DataFrame) -> pd.DataFrame:
    required = ['time', 'torque', 'rpm']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns: {missing}')

    out = df[required].copy()
    out = out.dropna(subset=required)
    out = out.sort_values('time', kind='mergesort').reset_index(drop=True)
    return out


def _cycle_info(df: pd.DataFrame, wheel_radius: float, gear_ratio: float) -> CycleInfo:
    df = _prepare_cycle(df)
    d_km = float(rpm2distance(df, wheel_radius=wheel_radius, gear_ratio=gear_ratio)[-1]) / 1000.0
    dur_s = elapsed_seconds(df)
    if d_km <= 0:
        raise ValueError('Cycle distance computed as <= 0 km. Check rpm/time.')
    if dur_s <= 0:
        raise ValueError('Cycle duration computed as <= 0 s. Check time.')
    return CycleInfo(df=df, distance_km=d_km, duration_s=dur_s)


def _slice_cycle_to_distance_km(
    cycle: pd.DataFrame,
    target_km: float,
    *,
    wheel_radius: float,
    gear_ratio: float,
) -> pd.DataFrame:
    """Prefix slice of cycle reaching target_km using distance integration + interpolation."""
    if target_km <= 0:
        return cycle.iloc[:1].copy()

    d_m = rpm2distance(cycle, wheel_radius=wheel_radius, gear_ratio=gear_ratio)
    target_m = target_km * 1000.0
    if target_m >= float(d_m[-1]):
        return cycle.copy()

    i = int(np.searchsorted(d_m, target_m, side='left'))
    i = max(1, min(i, len(cycle) - 1))

    seg = cycle.iloc[:i].copy()

    d0, d1 = float(d_m[i - 1]), float(d_m[i])
    denom = d1 - d0
    frac = 0.0 if denom <= 0 else float((target_m - d0) / denom)

    r0 = cycle.iloc[i - 1]
    r1 = cycle.iloc[i]
    last = {
        'time': float(r0['time'] + frac * (r1['time'] - r0['time'])),
        'torque': float(r0['torque'] + frac * (r1['torque'] - r0['torque'])),
        'rpm': float(r0['rpm'] + frac * (r1['rpm'] - r0['rpm'])),
    }
    return pd.concat([seg, pd.DataFrame([last])], ignore_index=True)


def _append_with_continuous_time(
    chunks: list[pd.DataFrame],
    chunk: pd.DataFrame,
    *,
    source: str,
    time_offset_s: float,
    drop_first_row: bool,
) -> float:
    c = chunk.copy()
    c['time'] = (c['time'].to_numpy(dtype=float) - float(c['time'].iloc[0])) + float(time_offset_s)
    c['source'] = source
    if drop_first_row and len(c) > 1:
        c = c.iloc[1:].reset_index(drop=True)
    chunks.append(c)
    return float(c['time'].iloc[-1])


def representative_time_series(
    autocross_data: pd.DataFrame,
    endurance_data: pd.DataFrame,
    lifecycle_distance_km: float,
    percent_autocross: float,
    percent_endurance: float,
    *,
    wheel_radius: float = 0.20574,
    gear_ratio: float = 2.14,
    interleave: bool = True,
) -> Tuple[pd.DataFrame, float]:
    """
    Copy + splice autocross/endurance cycles to reach a target lifecycle distance.

    - Uses full-cycle repetition plus a final partial cycle (interpolated) for any remainder.
    - Ensures output 'time' is continuous and starts at 0 s.

    Returns (synthetic_timeseries_df, lifecycle_time_hours).
    """
    if lifecycle_distance_km <= 0:
        raise ValueError('lifecycle_distance_km must be > 0')
    if percent_autocross < 0 or percent_endurance < 0:
        raise ValueError('percent_autocross/percent_endurance must be >= 0')
    if percent_autocross + percent_endurance <= 0:
        raise ValueError('percent_autocross + percent_endurance must be > 0')

    # normalize if not exactly 1.0
    s = float(percent_autocross + percent_endurance)
    p_auto = float(percent_autocross) / s
    p_end = float(percent_endurance) / s

    auto_km = float(lifecycle_distance_km) * p_auto
    end_km = float(lifecycle_distance_km) * p_end

    auto_info = _cycle_info(autocross_data, wheel_radius=wheel_radius, gear_ratio=gear_ratio)
    end_info = _cycle_info(endurance_data, wheel_radius=wheel_radius, gear_ratio=gear_ratio)

    chunks: list[pd.DataFrame] = []
    t_offset = 0.0
    first = True

    def add_cycle_portion(cycle_df: pd.DataFrame, name: str, portion_km: float) -> None:
        nonlocal t_offset, first
        if portion_km <= 1e-12:
            return

        info = auto_info if name == 'autocross' else end_info
        n_full = int(portion_km // info.distance_km)
        rem_km = float(portion_km - n_full * info.distance_km)

        for _ in range(n_full):
            t_offset = _append_with_continuous_time(
                chunks, cycle_df, source=name, time_offset_s=t_offset, drop_first_row=not first
            )
            first = False

        if rem_km > 1e-9:
            part = _slice_cycle_to_distance_km(
                cycle_df, rem_km, wheel_radius=wheel_radius, gear_ratio=gear_ratio
            )
            t_offset = _append_with_continuous_time(
                chunks, part, source=name, time_offset_s=t_offset, drop_first_row=not first
            )
            first = False

    if interleave:
        # Alternate full cycles to keep the output mixed; then append remaining partials.
        auto_full = int(auto_km // auto_info.distance_km)
        end_full = int(end_km // end_info.distance_km)

        for i in range(max(auto_full, end_full)):
            if i < auto_full:
                add_cycle_portion(auto_info.df, 'autocross', auto_info.distance_km)
                auto_km -= auto_info.distance_km
            if i < end_full:
                add_cycle_portion(end_info.df, 'endurance', end_info.distance_km)
                end_km -= end_info.distance_km

        add_cycle_portion(auto_info.df, 'autocross', auto_km)
        add_cycle_portion(end_info.df, 'endurance', end_km)
    else:
        add_cycle_portion(auto_info.df, 'autocross', auto_km)
        add_cycle_portion(end_info.df, 'endurance', end_km)

    synthetic_timeseries = pd.concat(chunks, ignore_index=True)
    synthetic_timeseries['time'] = synthetic_timeseries['time'] - float(synthetic_timeseries['time'].iloc[0])

    lifecycle_time_hours = float(synthetic_timeseries['time'].iloc[-1]) / SECONDS_PER_HOUR
    return synthetic_timeseries, lifecycle_time_hours

In [34]:
synthetic_timeseries, lifecycle_time_hours = representative_time_series(
    autocross_data=Autocross,
    endurance_data=Endurance,
    lifecycle_distance_km=lifecycle_distance_km,
    percent_autocross=percent_autocross,
    percent_endurance=percent_endurance,
)

print(f'Required run time: {lifecycle_time_hours:.3f} hours')
print('Synthetic rows:', len(synthetic_timeseries))

# quick checks
assert synthetic_timeseries['time'].iloc[0] == 0.0
assert (synthetic_timeseries['time'].diff().dropna() >= 0).all()

# Compute distance actually achieved
dist_km = float(rpm2distance(synthetic_timeseries)[-1]) / 1000.0
print(f'Achieved distance: {dist_km:.3f} km (target {lifecycle_distance_km:.3f} km)')

display(synthetic_timeseries.head())
display(synthetic_timeseries.tail())

Required run time: 20.864 hours
Synthetic rows: 749999
Achieved distance: 499.965 km (target 500.000 km)


,time,torque,rpm,source
0,0.000000,190.7500,315.0,autocross
1,0.040192,194.7500,337.0,autocross
2,0.081129,196.1875,365.0,autocross
3,0.121276,196.1875,390.0,autocross
4,0.159474,196.1875,415.0,autocross


,time,torque,rpm,source
749994,75081.500439,-0.000000,55.000000,endurance
749995,75081.540589,-0.000000,54.000000,endurance
749996,75081.580881,-0.000000,51.000000,endurance
749997,75081.621169,-0.000000,51.000000,endurance
749998,75110.592347,71.858802,58.984311,endurance


In [35]:
synthetic_timeseries['torque'] /= 2 #dual motors
#synthetic_timeseries = synthetic_timeseries.drop(columns=['source'], errors='ignore')
synthetic_timeseries = synthetic_timeseries[['time', 'rpm', 'torque']]
synthetic_timeseries.to_csv('EV27_500km_load_spectrum.csv', index=False)